# Bagging-based feature selection

This notebook finds a compact feature set for power-quality disturbance (PQD) classification. It follows the idea in `Vossi_01_Task.ipynb`—start with all eight features and remove features that do not improve classification—but evaluates the decision more reliably across bootstrap samples and multiple classifiers.

The word *bagging* here refers to bootstrap aggregation of the evaluation: each candidate feature set is trained on a sample drawn with replacement and scored on its out-of-bag (OOB) samples. Repeating this process reduces the chance that one train/test split determines the selected features. The final recommendation is a majority vote across the classifiers.


## 1. Imports and paths

The helper module contains the reusable data-loading and feature-selection code. The path setup makes the notebook work both when opened from `Code/01 Task` and when started from the repository root.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

TASK_DIR = Path.cwd()
if TASK_DIR.name != "01 Task":
    TASK_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if path.name == "01 Task")
sys.path.insert(0, str(TASK_DIR))

from bagging_feature_selection import (
    FEATURES,
    load_dataset,
    run_feature_selection,
)

## 2. Load and inspect the data

Each CSV file represents one disturbance class; its filename supplies the class label. The loader standardizes the column naming to `F1` through `F8`, adds the noise-level source, and combines all requested SNR folders into one table.

The feature names retain the ordering used by the original notebook. Keeping the noise-level column is useful for diagnostics, but it is not passed to the classifiers.


In [ ]:
folders = [TASK_DIR / name for name in (
    "SNR_20db", "SNR_30db", "SNR_40db", "SNR_50db", "SNR_noiseless"
)]
data = load_dataset(folders)
print(f"Loaded {len(data):,} samples from {data['noise_level'].nunique()} noise levels")
data.head()

## 3. Select features for every classifier

For each classifier, the selector starts with all eight features. It evaluates every possible one-feature removal using bootstrap OOB macro-F1, accepts the best removal when it is not meaningfully worse than the current set, and repeats until no further removal is supported or one feature remains.

The scaler is inside the estimator pipeline, so it is fitted separately for every bootstrap training sample. This prevents validation/OOB information from leaking into preprocessing. `min_improvement=0.002` acts as a small tolerance: tiny score fluctuations do not cause an unstable feature removal. Increase `n_bootstraps` for more stable results in a final experiment.


In [ ]:
results, overall_features = run_feature_selection(
    data,
    n_bootstraps=20,
    min_improvement=0.002,
    random_state=42,
)

summary = pd.DataFrame({
    "classifier": [result.classifier for result in results],
    "selected_features": [list(result.selected_features) for result in results],
    "bootstrap_macro_f1": [result.score for result in results],
})
summary

## 4. Build the overall recommendation

A feature is included in `overall_features` when at least half of the classifiers selected it. This consensus is deliberately model-agnostic: features that are useful for only one model do not automatically become the project-wide recommendation. If no feature reaches the majority threshold, the most frequently selected feature is retained as a safe non-empty fallback.


In [ ]:
print("Overall feature set:", list(overall_features))
print("Use these features for downstream classifier training:", list(overall_features))

## 5. Interpret the output

The summary reports the selected subset and mean bootstrap macro-F1 for each classifier. Macro-F1 gives every PQD class equal weight, which is appropriate when performance should not be dominated by the most common class.

Use `overall_features` for downstream training and report the per-classifier subsets and scores as evidence of selection stability. For a final reported result, rerun with a larger `n_bootstraps` value and keep the random seed fixed so the experiment remains reproducible.


## Reading the result

`summary` shows each model's selected subset and its mean out-of-bag macro-F1. `overall_features` contains features selected by at least half of the classifiers; if the vote is empty, the most frequently selected feature is retained. Increase `n_bootstraps` for a more stable final experiment.